# 03 — Modelagem e Avaliação

Este notebook realiza o treinamento e a avaliação dos modelos de aprendizado de máquina utilizados para classificar a direção do Ibovespa cinco pregões à frente.

Modelos avaliados:
- Baseline Majority
- Baseline Persistence
- Regressão Logística
- Árvore de Decisão
- Random Forest

A divisão dos dados respeita a ordem cronológica da série temporal, evitando vazamento de informações futuras.

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.metrics import (
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

import json

In [3]:
# Caminho do dataset processado
arquivo_dataset = Path("../data/processed/ibovespa_features.csv")

# Carregar dados
df = pd.read_csv(
    arquivo_dataset,
    parse_dates=["Date"]
)

print("Dimensões: ", df.shape)

display(df.head())

Dimensões:  (3950, 9)


,Date,Retorno_Diario,Retorno_5d,Retorno_10d,Media_Retorno_5d,Volatilidade_5d,Variacao_Intradiaria,Amplitude_Diaria,Direcao_Futura
0,2010-01-18,0.613239,-1.465222,-0.919409,-0.292293,0.787222,0.605946,1.108969,0
1,2010-01-19,0.731978,-0.238313,-0.471241,-0.044524,0.890929,0.739236,1.697475,0
2,2010-01-20,-2.444607,-3.104355,-3.575620,-0.621636,1.326126,-2.433442,3.486513,0
3,2010-01-21,-2.829912,-5.058667,-5.934621,-1.021674,1.663390,-2.829912,3.730529,0
4,2010-01-22,-0.075449,-3.998376,-5.754095,-0.800950,1.709857,-0.066401,1.856521,1


In [8]:
print("### Validação do dataset ###")

print("\nDimensões:", df.shape)

print("\nPeríodo:")
print("Inicial:", df["Date"].min())
print("Final:", df["Date"].max())

print("\nValores ausentes:")
print(df.isna().sum())

print("\nValores infinitos:")
print(
    np.isinf(
        df.select_dtypes(include="number")
    ).sum()
)

print("\nDistribuição do target:")
print(
    df["Direcao_Futura"]
    .map({0: "QUEDA", 1: "ALTA"})
    .value_counts()
)

### Validação do dataset ###

Dimensões: (3950, 9)

Período:
Inicial: 2010-01-18 00:00:00
Final: 2025-12-19 00:00:00

Valores ausentes:
Date                    0
Retorno_Diario          0
Retorno_5d              0
Retorno_10d             0
Media_Retorno_5d        0
Volatilidade_5d         0
Variacao_Intradiaria    0
Amplitude_Diaria        0
Direcao_Futura          0
dtype: int64

Valores infinitos:
Retorno_Diario          0
Retorno_5d              0
Retorno_10d             0
Media_Retorno_5d        0
Volatilidade_5d         0
Variacao_Intradiaria    0
Amplitude_Diaria        0
Direcao_Futura          0
dtype: int64

Distribuição do target:
Direcao_Futura
ALTA     2082
QUEDA    1868
Name: count, dtype: int64


In [10]:
# Definição das features e target que o modelo usará pra aprender e pra prever

features = [
    "Retorno_Diario",
    "Retorno_5d",
    "Retorno_10d",
    "Media_Retorno_5d",
    "Volatilidade_5d",
    "Variacao_Intradiaria",
    "Amplitude_Diaria",
]

target = "Direcao_Futura"

X = df[features].copy()
y = df[target].copy()

print("Features:")
print(features)

print("\nDimensão de X:", X.shape)
print("\nDimensão de y:", y.shape)

Features:
['Retorno_Diario', 'Retorno_5d', 'Retorno_10d', 'Media_Retorno_5d', 'Volatilidade_5d', 'Variacao_Intradiaria', 'Amplitude_Diaria']

Dimensão de X: (3950, 7)

Dimensão de y: (3950,)


In [12]:
# Separação das bases de treino, validação e teste

# Total de observações
n = len(df)

# Definir os pontos de corte
i_train = int(n * 0.70)
i_val = int(n * 0.85)

# Divisão cronologica
X_train = X.iloc[:i_train]
y_train = y.iloc[:i_train]

X_val = X.iloc[i_train:i_val]
y_val = y.iloc[i_train:i_val]

X_test = X.iloc[i_val:]
y_test = y.iloc[i_val:]

print("### Dimensões ###")
print("Treinamento:", X_train.shape)
print("Validação:", X_val.shape)
print("Teste:", X_test.shape)

### Dimensões ###
Treinamento: (2765, 7)
Validação: (592, 7)
Teste: (593, 7)


In [14]:
# Conferencia das datas de cada conjunto
print("### Períodos ###")

print(
    "\nTreinamento:",
    df["Date"].iloc[0],
    "→",
    df["Date"].iloc[i_train - 1]
)

print(
    "Validação:",
    df["Date"].iloc[i_train],
    "→",
    df["Date"].iloc[i_val - 1]
)

print(
    "Teste:",
    df["Date"].iloc[i_val],
    "→",
    df["Date"].iloc[-1]
)

### Períodos ###

Treinamento: 2010-01-18 00:00:00 → 2021-03-25 00:00:00
Validação: 2021-03-26 00:00:00 → 2023-08-08 00:00:00
Teste: 2023-08-09 00:00:00 → 2025-12-19 00:00:00


In [15]:
def mostrar_distribuicao(nome, y):
    distribuicao = (
        y.map({0: "QUEDA", 1: "ALTA"})
        .value_counts(normalize=True)
        .mul(100)
        .round(2)
    )

    print(f"\n{nome}:")
    print(distribuicao)


mostrar_distribuicao("Treinamento", y_train)
mostrar_distribuicao("Validação", y_val)
mostrar_distribuicao("Teste", y_test)


Treinamento:
Direcao_Futura
ALTA     53.78
QUEDA    46.22
Name: proportion, dtype: float64

Validação:
Direcao_Futura
QUEDA    51.86
ALTA     48.14
Name: proportion, dtype: float64

Teste:
Direcao_Futura
ALTA     52.28
QUEDA    47.72
Name: proportion, dtype: float64


# Antes dos modelos de ML, precisamos estabelecer uma referência.

O Majority Baseline prevê a classe mais frequente no conjunto de treinamento para todas as observações futuras.

Por exemplo, se:
Treinamento:
```
ALTA   55%
QUEDA  45%
```
o baseline simplesmente diz:
"Vou prever ALTA para todo mundo."

Isso parece extremamente simples — e é justamente essa a ideia.

In [28]:
# Implementação do Majority Baseline

# Identificar a classe majoritária no treinamento
classe_majoritaria = y_train.mode()[0]

print("Classe majoritária:", classe_majoritaria)

# Prever a classe majoritária para validação e teste
y_val_majority = np.full(
    len(y_val),
    classe_majoritaria
)

y_test_majority = np.full(
    len(y_test),
    classe_majoritaria
)

Classe majoritária: 1


In [26]:
# Função de avaliação do modelo
def avaliar_modelo(nome, y_true, y_pred):
    return {
        "model": nome,
        "balanced_accuracy": balanced_accuracy_score(
            y_true,
            y_pred
        ),
        "precision_high": precision_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),
        "recall_high": recall_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        ),
        "f1_high": f1_score(
            y_true,
            y_pred,
            pos_label=1,
            zero_division=0
        )
    }

In [31]:

resultado_majority = avaliar_modelo(
    "Majority",
    y_test,
    y_test_majority
)

print(json.dumps(resultado_majority, ensure_ascii=False, indent=4))

{
    "model": "Majority",
    "balanced_accuracy": 0.5,
    "precision_high": 0.5227655986509275,
    "recall_high": 1.0,
    "f1_high": 0.6866002214839424
}
